# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python) 

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [ ]:
# importar librerías
import pandas as pd

In [ ]:
# cargar archivos

orders = pd.read_csv(
    'https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv'
)

catalog = pd.read_csv(
    'https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv'
)

marketing = pd.read_csv(
    'https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv'
)


In [ ]:
# explorar datasets

print('ORDERS')
display(orders.head())

print('CATALOG')
display(catalog.head())

print('MARKETING')
display(marketing.head())

---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas 

---

In [ ]:
print('ORDERS')
orders.info()

print('\nCATALOG')
catalog.info()

print('\nMARKETING')
marketing.info()

In [ ]:
print('Valores ausentes en orders:')
print(orders.isna().sum())

print('\nValores ausentes en catalog:')
print(catalog.isna().sum())

print('\nValores ausentes en marketing:')
print(marketing.isna().sum())

In [ ]:
# Convertir columnas de fecha
orders['fecha_hora_pedido'] = pd.to_datetime(orders['fecha_hora_pedido'])
marketing['fecha'] = pd.to_datetime(marketing['fecha'])

In [ ]:

print(orders['fecha_hora_pedido'].dtype)
print(marketing['fecha'].dtype)


In [ ]:
# Revisar registros duplicados
print('Duplicados en orders:', orders.duplicated().sum())
print('Duplicados en catalog:', catalog.duplicated().sum())
print('Duplicados en marketing:', marketing.duplicated().sum())

In [ ]:
# Eliminar duplicados de pedidos
orders = orders.drop_duplicates().reset_index(drop=True)

In [ ]:
print('Duplicados restantes en orders:', orders.duplicated().sum())

In [ ]:
# Revisar valores de variables categóricas
columnas_categoricas_orders = [
    'pais',
    'dispositivo',
    'fuente_referencia',
    'nombre_producto',
    'categoria_producto'
]

for columna in columnas_categoricas_orders:
    print(f'\n{columna}:')
    print(orders[columna].value_counts(dropna=False))

In [ ]:
# Estandarizar nombres de países
orders['pais'] = orders['pais'].str.capitalize()
marketing['pais'] = marketing['pais'].str.capitalize()

In [ ]:
print(orders['pais'].value_counts(dropna=False))

In [ ]:
#revisar variables numericas
display(
    orders[
        ['cantidad',
         'precio_unitario',
         'monto_descuento',
         'monto_total']
    ].describe()
)

In [ ]:
print(
    orders['cantidad']
    .value_counts(dropna=False)
    .sort_index()
)

**Observación:** La mayoría de los pedidos contienen 1 o 2 unidades. Se identificaron algunos valores negativos y cantidades de 10,000 y 20,000, que se consideran inconsistentes con el resto de los datos y se excluyen del análisis.

In [ ]:
orders = orders[
    (orders['cantidad'] > 0) &
    (orders['cantidad'] <= 2)
].copy()

In [ ]:
# Tratar valores faltantes
orders['pais'] = orders['pais'].fillna('Desconocido')
orders['dispositivo'] = orders['dispositivo'].fillna('desconocido')
orders['fuente_referencia'] = orders['fuente_referencia'].fillna('desconocido')
orders['nombre_producto'] = orders['nombre_producto'].fillna('Producto desconocido')
orders['categoria_producto'] = orders['categoria_producto'].fillna('Desconocido')

marketing['canal'] = marketing['canal'].fillna('desconocido')

In [ ]:
# Validacion final
orders = orders.reset_index(drop=True)

In [ ]:
print('Valores ausentes en orders:')
print(orders.isna().sum())

print('\nDuplicados en orders:')
print(orders.duplicated().sum())

In [ ]:
display(
    orders[
        ['cantidad',
         'precio_unitario',
         'monto_descuento',
         'monto_total']
    ].describe()
)

---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

---

In [ ]:
# Exportar datasets limpios
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

In [ ]:
print('orders_clean:', orders.shape)
print('catalog_clean:', catalog.shape)
print('marketing_clean:', marketing.shape)


In [ ]:
print('\nValores ausentes en marketing:')
print(marketing.isna().sum())

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)? 
- ¿Cuál es el costo total? 
- ¿Cuánto se ha invertido en marketing? 
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden? 
- ¿Cuál es la cantidad promedio de productos por orden? 
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal? 

In [ ]:

# Unir pedidos con el catálogo para obtener el costo unitario
orders_costos = orders.merge(
    catalog[['nombre_producto', 'costo_unitario']],
    on='nombre_producto',
    how='left'
)

In [ ]:
print('Pedidos sin costo asociado:')
print(orders_costos['costo_unitario'].isna().sum())

Observación: Algunos pedidos no tienen un producto identificado, por lo que no es posible asociarlos con un costo unitario del catálogo. Estos casos se mantienen en el cálculo de ingresos, pero no aportan al costo total.

In [ ]:
# Calcular costo por pedido
orders_costos['costo_total'] = (
    orders_costos['cantidad'] * orders_costos['costo_unitario']
)

In [ ]:
# Calcular KPIs de rentabilidad
revenue_total = orders['monto_total'].sum()
costo_total = orders_costos['costo_total'].sum()
gasto_marketing = marketing['gasto'].sum()

profit = revenue_total - costo_total - gasto_marketing

In [ ]:
print(f'Revenue total: {revenue_total:,.2f}')
print(f'Costo total: {costo_total:,.2f}')
print(f'Gasto en marketing: {gasto_marketing:,.2f}')
print(f'Profit: {profit:,.2f}')


**Conclusión:** 

RappiPlus presenta una ganancia estimada positiva de aproximadamente 2.92 millones, después de descontar los costos de productos registrados y la inversión en marketing. Sin embargo, este resultado no incluye otros gastos operativos, por lo que no representa la utilidad neta definitiva del negocio.


In [ ]:
ticket_promedio = orders['monto_total'].mean()

print(f'Ticket promedio por orden: {ticket_promedio:,.2f}')

In [ ]:
cantidad_promedio = orders['cantidad'].mean()

print(f'Cantidad promedio de productos por orden: {cantidad_promedio:.2f}')

In [ ]:
productos_vendidos = (
    orders.groupby('nombre_producto')['cantidad']
    .sum()
    .sort_values(ascending=False)
)

print(productos_vendidos)

In [ ]:
print('Producto más vendido:')
print(productos_vendidos.head(1))

In [ ]:
gasto_por_canal = (
    marketing.groupby('canal')['gasto']
    .sum()
    .sort_values(ascending=False)
)

print(gasto_por_canal)

In [ ]:
# Rentabilidad por categoría de producto

rentabilidad_categoria = (
    orders_costos.groupby('categoria_producto')
    .agg(
        revenue=('monto_total', 'sum'),
        costo=('costo_total', 'sum')
    )
)

rentabilidad_categoria['profit'] = (
    rentabilidad_categoria['revenue']
    - rentabilidad_categoria['costo']
)

rentabilidad_categoria = rentabilidad_categoria.sort_values(
    'profit',
    ascending=False
)

rentabilidad_categoria

In [ ]:
rentabilidad_categoria['margen_pct'] = (
    rentabilidad_categoria['profit']
    / rentabilidad_categoria['revenue']
    * 100
)

rentabilidad_categoria

**Conclusión por segmento:** 

Entre las categorías con información de costos disponible, Hogar presenta la mayor rentabilidad, con una ganancia de productos aproximada de 2.02 millones y un margen de 62.46 %. Moda y Electrónica también presentan ganancias positivas, aunque con márgenes ligeramente menores. La categoría Desconocido no se considera para comparar la rentabilidad, ya que sus registros no tienen un costo unitario asociado y, por ello, el margen calculado no es representativo.


### Análisis adicional: Pedidos con pérdidas

Aunque el negocio presenta una ganancia general positiva, se revisará si existen pedidos individuales cuyos costos superen los ingresos.

In [ ]:
# Calcular ganancia por pedido
orders_costos['profit_pedido'] = (
    orders_costos['monto_total'] - orders_costos['costo_total']
)

orders_costos['profit_pedido'].describe()

In [ ]:
# Identificar pedidos con pérdidas
pedidos_perdida = orders_costos[
    orders_costos['profit_pedido'] < 0
]

display(pedidos_perdida.head())

In [ ]:
# Considerar únicamente pedidos con costos conocidos
pedidos_con_costo = orders_costos[
    orders_costos['costo_total'].notna()
]

cantidad_perdida = len(pedidos_perdida)

porcentaje_perdida = (
    cantidad_perdida / len(pedidos_con_costo) * 100
)

print(f'Pedidos con pérdidas: {cantidad_perdida}')
print(f'Porcentaje de pedidos con pérdidas: {porcentaje_perdida:.2f}%')

In [ ]:
# Productos con mayor número de pedidos con pérdidas
perdidas_producto = (
    pedidos_perdida.groupby('nombre_producto')
    .size()
    .sort_values(ascending=False)
)

display(perdidas_producto)

In [ ]:
import matplotlib.pyplot as plt

perdidas_producto.plot(
    kind='barh',
    figsize=(8, 4)
)

plt.title('Pedidos con pérdidas por producto')
plt.xlabel('Número de pedidos')
plt.ylabel('Producto')

plt.tight_layout()
plt.show()

**Hallazgo de pedidos con pérdidas:**

De los 24,906 pedidos con costos identificados, 4,528 (18.18 %) presentan pérdidas. Estas operaciones se concentran principalmente en Jacket-Winter-M, Laptop-Gaming-16GB y Blender-XL-Red. Aunque el negocio genera ganancias a nivel general, estos resultados indican que algunos productos requieren una revisión de su rentabilidad individual.


**Hallazgos principales:**

El negocio presenta una ganancia estimada positiva de aproximadamente 2.92 millones, después de descontar los costos de productos registrados y la inversión en marketing. El ticket promedio es de 385.88 y cada pedido contiene, en promedio, 1.5 productos.

Vacuum-Pro-Black es el producto con mayor número de unidades vendidas, aunque los siguientes productos presentan cifras muy similares. El canal con mayor inversión registrada es social. Entre las categorías con costos conocidos, Hogar presenta el mayor margen de productos, con un 62.46 %.

Adicionalmente, se identificó que el 18.18 % de los pedidos con costos conocidos presenta pérdidas antes de descontar marketing. Estas operaciones se concentran principalmente en tres productos, por lo que convendría revisar sus precios y costos para comprender las diferencias de rentabilidad.


---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?


In [ ]:
import pandas as pd
from sqlalchemy import create_engine
import os

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': os.environ['RAPPIPLUS_DB_PASSWORD'],
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [ ]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

In [ ]:
# PARTE 1: Totales del funnel
# ======================

query_totals = '''
SELECT
    nombre_evento,
    COUNT(DISTINCT id_usuario) AS usuarios
FROM events
GROUP BY nombre_evento;
'''

totals = pd.read_sql(query_totals, con=engine)
totals

In [ ]:
# PARTE 2: Conversiones
# ======================

query_conversion = '''
WITH funnel AS (
    SELECT
        nombre_evento,
        COUNT(DISTINCT id_usuario) AS usuarios,
        CASE
            WHEN nombre_evento = 'first_visit' THEN 1
            WHEN nombre_evento = 'select_item' THEN 2
            WHEN nombre_evento = 'add_to_cart' THEN 3
            WHEN nombre_evento = 'begin_checkout' THEN 4
            WHEN nombre_evento = 'add_payment_info' THEN 5
            WHEN nombre_evento = 'purchase' THEN 6
        END AS orden
    FROM events
    GROUP BY nombre_evento
)

SELECT
    nombre_evento,
    usuarios,
    LAG(usuarios) OVER (ORDER BY orden) AS usuarios_paso_anterior,
    ROUND(
        usuarios * 100.0 /
        LAG(usuarios) OVER (ORDER BY orden),
        2
    ) AS tasa_conversion
FROM funnel
ORDER BY orden;
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

In [ ]:
query_validacion = '''
SELECT COUNT(DISTINCT id_usuario) AS usuarios_sin_select_item
FROM events
WHERE nombre_evento = 'add_to_cart'
AND id_usuario NOT IN (
    SELECT id_usuario
    FROM events
    WHERE nombre_evento = 'select_item'
);
'''

validacion = pd.read_sql(query_validacion, con=engine)
validacion

**Conclusión:**

La mayor disminución de usuarios se observa entre begin_checkout y add_payment_info, con una pérdida de 958 usuarios (13.29 % respecto a la etapa anterior). Esto sugiere que el proceso de pago es un punto importante para investigar posibles oportunidades de mejora.

La proporción entre los usuarios que registraron first_visit y purchase es de aproximadamente 80.04 %.

También se identificó que add_to_cart registra más usuarios que select_item, con una tasa calculada de 100.69 %. Al revisar los datos, encontramos 401 usuarios que agregaron productos al carrito sin registrar un evento select_item.

Esto podría deberse a que los usuarios pueden agregar productos directamente al carrito sin seleccionar previamente un artículo de una lista, aunque también podrían existir diferencias en el registro de eventos. Con la información disponible no es posible confirmar la causa.

Por lo tanto, las tasas calculadas representan comparaciones entre usuarios únicos de cada evento, pero no necesariamente un recorrido secuencial completo.

Recomendación: Investigar qué sucede entre el inicio del checkout y el ingreso de la información de pago, ya que es la etapa donde se observa la mayor disminución de usuarios. También sería conveniente revisar cómo se registran los eventos para comprender mejor los distintos recorridos de compra y obtener tasas de conversión más precisas.

---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users` 
- `user_activity` 

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [ ]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

In [ ]:
# Explorar tabla user_activity
# =========================

query_user_activity = '''
SELECT *
FROM user_activity;
'''

user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

In [ ]:
users.head()

In [ ]:
user_activity.head()

In [ ]:
users.info()
user_activity.info()

In [ ]:

# Retención por cohortes
# ======================

query_cohort_retention_final = '''
WITH cohortes AS (
    SELECT
        id_usuario,
        DATE_TRUNC('month', CAST(fecha_registro AS DATE)) AS cohorte
    FROM users
)

SELECT
    c.cohorte,
    COUNT(DISTINCT c.id_usuario) AS usuarios_cohorte,

    COUNT(DISTINCT CASE
        WHEN ua.dias_despues_registro = 7 AND ua.activo = 1
        THEN c.id_usuario
    END) AS retenido_w1,

    COUNT(DISTINCT CASE
        WHEN ua.dias_despues_registro = 14 AND ua.activo = 1
        THEN c.id_usuario
    END) AS retenido_w2,

    COUNT(DISTINCT CASE
        WHEN ua.dias_despues_registro = 21 AND ua.activo = 1
        THEN c.id_usuario
    END) AS retenido_w3,

    ROUND(
        COUNT(DISTINCT CASE
            WHEN ua.dias_despues_registro = 7 AND ua.activo = 1
            THEN c.id_usuario
        END) * 100.0 / COUNT(DISTINCT c.id_usuario),
        2
    ) AS semana_1,

    ROUND(
        COUNT(DISTINCT CASE
            WHEN ua.dias_despues_registro = 14 AND ua.activo = 1
            THEN c.id_usuario
        END) * 100.0 / COUNT(DISTINCT c.id_usuario),
        2
    ) AS semana_2,

    ROUND(
        COUNT(DISTINCT CASE
            WHEN ua.dias_despues_registro = 21 AND ua.activo = 1
            THEN c.id_usuario
        END) * 100.0 / COUNT(DISTINCT c.id_usuario),
        2
    ) AS semana_3

FROM cohortes c
LEFT JOIN user_activity ua
    ON c.id_usuario = ua.id_usuario

GROUP BY c.cohorte
ORDER BY c.cohorte;
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

**Conclusión:**

La retención se mantiene relativamente estable durante las primeras tres semanas posteriores al registro, con porcentajes cercanos al 40–44 % en todas las cohortes. No se observa una disminución continua, lo que sugiere que algunos usuarios podrían regresar después de periodos de inactividad.

La cohorte de febrero presenta la mayor retención en la semana 3 (43.98 %), mientras que enero registra la menor (40.32 %).

Recomendación: Investigar por qué aproximadamente el 60 % de los usuarios no registra actividad en cada una de las semanas analizadas. Sería útil evaluar estrategias de reactivación y seguimiento durante las primeras semanas posteriores al registro

---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado** 
4. **Interpretar el resultado**  



### Hipótesis estadística

- **H₀ (Hipótesis nula):** No existe una diferencia significativa en la tasa de conversión entre el grupo control y el grupo tratamiento.
- **H₁ (Hipótesis alternativa):** Existe una diferencia significativa en la tasa de conversión entre el grupo control y el grupo tratamiento.

**Test estadístico:** z-test de proporciones  
**Nivel de significancia (alpha):** 0.05
  

In [ ]:

import pandas as pd
from statsmodels.stats.proportion import proportions_ztest


In [ ]:
# Cargar dataset del experimento
experiment = pd.read_csv(
    'https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv'
)

experiment.head()

In [ ]:
experiment.info()

Observación: La consigna se refiere a la métrica conversion, mientras que en el dataset la columna aparece como convirtio. Para mantener consistencia en el análisis, se renombra la columna.

In [ ]:
experiment = experiment.rename(
    columns={'convirtio': 'conversion'}
)

In [ ]:
print(experiment['variante'].value_counts())
print(experiment['conversion'].value_counts())

In [ ]:
print(
    experiment.groupby('variante')['conversion'].mean()
)

In [ ]:
control = experiment[
    experiment['variante'] == 'control'
]

tratamiento = experiment[
    experiment['variante'] == 'tratamiento'
]

In [ ]:
conversiones = [
    control['conversion'].sum(),
    tratamiento['conversion'].sum()
]

totales = [
    len(control),
    len(tratamiento)
]

In [ ]:
z_stat, p_value = proportions_ztest(
    conversiones,
    totales
)

print(f'Estadístico z: {z_stat:.3f}')
print(f'Valor p: {p_value:.4f}')

In [ ]:
tasa_control = control['conversion'].mean()
tasa_tratamiento = tratamiento['conversion'].mean()

print(f'Tasa de conversión control: {tasa_control:.2%}')
print(f'Tasa de conversión tratamiento: {tasa_tratamiento:.2%}')

In [ ]:
alpha = 0.05

if p_value < alpha:
    print('Se rechaza la hipótesis nula.')
    print('Existe una diferencia estadísticamente significativa entre las tasas de conversión.')
else:
    print('No se rechaza la hipótesis nula.')
    print('No hay evidencia suficiente de una diferencia significativa entre las tasas de conversión.')

### Conclusión del experimento

La tasa de conversión del grupo control fue de 15.69 %, mientras que la del grupo tratamiento fue de 16.29 %.

El valor p obtenido fue de 0.4161. Como este valor es mayor que el nivel de significancia de 0.05, no se rechaza la hipótesis nula.

Por lo tanto, no existe evidencia estadística suficiente para concluir que la modificación en la interfaz del checkout haya generado un impacto significativo en la tasa de conversión.

**Recomendación:** No implementar el cambio únicamente con base en este experimento. Aunque el grupo tratamiento presenta una conversión ligeramente mayor, la diferencia puede deberse al azar. Sería recomendable continuar evaluando la modificación antes de tomar una decisión definitiva.

---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión. 

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

### Preparación y modelo de datos

Para construir el dashboard se utilizaron los tres archivos limpios del Paso 1: `orders_clean.csv`, `catalog_clean.csv` y `marketing_clean.csv`.

En Power BI se relacionó la tabla de pedidos con el catálogo de productos para incorporar los costos unitarios. También se crearon las dimensiones `dim_fecha` y `dim_pais`, que permiten filtrar las ventas y la inversión en marketing por periodo y país.

Las relaciones se configuraron de uno a muchos, con dirección de filtro simple desde las dimensiones hacia las tablas correspondientes.

Se crearon medidas DAX para calcular ingresos, costos, inversión en marketing, ticket promedio, productos por pedido y rentabilidad.

Para el análisis se diferenciaron dos indicadores:

* **Ganancia de productos:** ingresos menos costos de los productos registrados.
* **Ganancia estimada:** ingresos menos costos de productos registrados e inversión en marketing.

Esta distinción es importante porque no se cuenta con información para distribuir el gasto de marketing entre productos o categorías.

La ganancia estimada tampoco incluye otros gastos operativos, por lo que no debe interpretarse como la utilidad neta definitiva del negocio.

---

### Overview Ejecutivo

La primera página reúne los indicadores generales de RappiPlus y permite revisar cómo cambian las ventas y la rentabilidad durante el primer semestre de 2025.

Se incluyeron cinco indicadores:

| Indicador              | Resultado general |
| ---------------------- | ----------------: |
| Ingresos totales       |     $9,622,281.56 |
| Ganancia estimada      |     $2,921,569.02 |
| Inversión en marketing |     $2,871,843.53 |
| Ticket promedio        |           $385.88 |
| Productos por pedido   |              1.50 |

Para complementar estos resultados se utilizaron tres visualizaciones:

* **Evolución mensual de ingresos:** muestra las variaciones de las ventas entre enero y junio.
* **Ingresos y ganancia por categoría:** permite comparar las categorías considerando tanto sus ingresos como la ganancia de productos antes de marketing.
* **Evolución mensual del margen estimado:** muestra qué porcentaje de los ingresos permanece después de descontar los costos considerados.

Se incorporaron filtros por país y mes para explorar los resultados bajo diferentes condiciones.

También se añadieron dos indicadores mediante información sobre herramientas (*tooltips*):

* **Ingresos YTD:** muestra las ventas acumuladas desde enero hasta el mes consultado.
* **Margen de productos:** permite conocer el porcentaje de ganancia de cada categoría antes de marketing.

El cálculo YTD se realizó utilizando la tabla calendario y se comprobó que funciona tanto para el conjunto completo como al seleccionar un país y un mes específicos.

En el gráfico de categorías se mantuvo la comparación independiente de los indicadores generales. Esto evita atribuir a una categoría la inversión total de marketing, ya que no se dispone de esa distribución.

**Observaciones**

RappiPlus registró ingresos de 9.62 millones y una ganancia estimada de 2.92 millones durante el periodo analizado.

Las tres categorías presentan ingresos relativamente similares, pero existen diferencias en sus márgenes de productos. Hogar registra el margen más alto, con 62.46 %, seguido de Moda (59.32 %) y Electrónica (58.66 %).

Esto muestra que no basta con comparar las ventas: también es necesario considerar cuánto cuesta generar esos ingresos.

---

### Detalle de Ventas

La segunda página se diseñó para consultar los productos y pedidos individuales.

Se incluyó un gráfico de unidades vendidas por producto y una tabla con el identificador del pedido, fecha, producto, cantidad, ingresos, costo y ganancia de productos.

La tabla utiliza formato condicional para distinguir las ganancias positivas de las pérdidas y se ordenó inicialmente de menor a mayor ganancia. De esta manera, los pedidos con mayores pérdidas aparecen primero.

También se incorporaron filtros por fecha y categoría. Al seleccionar un producto en el gráfico de barras, la tabla muestra los pedidos relacionados.

Se optó por mantener esta interacción dentro de la misma página, sin crear una vista adicional de *drill-through*.

La tabla presenta la ganancia de productos antes de marketing. Esta información permite revisar la rentabilidad de las operaciones sin mezclarla con los gastos de marketing, que se analizan a nivel general.

**Observaciones**

Los productos con mayor número de unidades vendidas presentan cantidades muy cercanas. Vacuum-Pro-Black registra 6,284 unidades, seguido de Blender-XL-Red con 6,279 y Jacket-Winter-M con 6,256.

Por otra parte, el análisis realizado en Python identificó 4,528 pedidos con pérdidas entre los 24,906 pedidos con costos conocidos.

La tabla de Power BI permite consultar estas operaciones y revisar sus ingresos y costos, complementando el análisis de rentabilidad realizado anteriormente.

---

### Narrativa ejecutiva (SCQA)

**Situación**

RappiPlus presenta ingresos y una ganancia estimada positiva durante el primer semestre de 2025.

**Complicación**

Aunque el resultado general es favorable, la rentabilidad no es igual entre categorías y existen pedidos cuyos costos superan los ingresos generados.

**Pregunta**

¿Qué productos y categorías conviene revisar para mejorar la rentabilidad del negocio?

**Respuesta**

El análisis muestra que Hogar tiene el mayor margen de productos, mientras que ciertas operaciones presentan pérdidas.

Antes de proponer cambios de precios, sería conveniente revisar los costos y márgenes de los productos que concentran estos pedidos.

Los filtros del dashboard permiten explorar las diferencias por país y periodo, mientras que la tabla detallada ayuda a identificar operaciones específicas que requieren atención.

---

### Conclusión del dashboard

Las dos páginas permiten revisar la rentabilidad de RappiPlus desde diferentes niveles.

El Overview Ejecutivo muestra los resultados generales y su evolución, mientras que Detalle de Ventas permite consultar los productos y pedidos que contribuyen a esos resultados.

La principal utilidad del dashboard es que no se limita a mostrar cuánto vende el negocio, sino que también permite revisar sus costos y localizar operaciones con pérdidas.

**Limitaciones:** la ganancia estimada considera únicamente los costos de productos registrados y la inversión en marketing. Existen 30 pedidos sin costos identificables, por lo que la estimación puede estar sobrevalorada. Además, los archivos no especifican una moneda común para los tres países; el símbolo `$` se utiliza como formato de presentación.


##  Conclusiones generales y recomendaciones

El objetivo del proyecto fue analizar el desempeño de RappiPlus desde distintas perspectivas: calidad de los datos, ventas, rentabilidad, comportamiento de los usuarios y resultados de un experimento A/B.

Después de realizar los análisis, encontré que el negocio presenta resultados financieros positivos, pero también existen aspectos que conviene investigar antes de tomar decisiones comerciales.

### ¿Los usuarios realmente compran más?

Durante el primer semestre de 2025, RappiPlus registró ingresos de 9.62 millones, con un ticket promedio de 385.88 y aproximadamente 1.5 productos por pedido.

Estos resultados permiten conocer el comportamiento general de las ventas, pero no son suficientes para afirmar que los usuarios están comprando más que antes. Para comprobarlo sería necesario comparar el comportamiento de compra entre periodos o grupos equivalentes.

El análisis de retención aporta información adicional sobre la actividad de los usuarios. Durante las primeras tres semanas posteriores al registro, las cohortes presentan porcentajes cercanos al 40–44 %, sin una disminución continua.

Esto indica que una parte importante de los usuarios no registra actividad en cada semana analizada. Sin embargo, no significa necesariamente que hayan abandonado la plataforma, ya que pueden regresar después de periodos de inactividad.

###  ¿El modelo está generando ganancias?

Con los costos disponibles, RappiPlus presenta una ganancia estimada positiva de 2.92 millones, después de descontar los costos de productos registrados y la inversión en marketing.

Sin embargo, al revisar los pedidos individualmente encontré que 4,528 de los 24,906 pedidos con costos conocidos presentan pérdidas, equivalentes al 18.18 %.

Estos pedidos se concentran principalmente en Jacket-Winter-M, Laptop-Gaming-16GB y Blender-XL-Red.

Este hallazgo es importante porque muestra que una ganancia general positiva no significa que todas las operaciones sean rentables.

Antes de realizar ajustes comerciales, recomendaría revisar los costos y precios de estos productos. También sería útil calcular qué proporción de sus pedidos presenta pérdidas, para distinguir entre productos con muchas ventas y productos con una mayor frecuencia de operaciones no rentables.

### ¿Se están perdiendo oportunidades durante el proceso de compra?

En el análisis del embudo, la mayor diferencia entre etapas consecutivas se observó entre `begin_checkout` y `add_payment_info`, con 958 usuarios menos, equivalentes al 13.29 % respecto a quienes registraron el inicio del checkout.

También se identificaron 401 usuarios con eventos `add_to_cart` que no registraron un evento `select_item`.

Estos resultados sugieren que vale la pena investigar el proceso de compra y revisar cómo se registran los eventos.

Sin embargo, los conteos corresponden a usuarios únicos por evento y no necesariamente representan un recorrido secuencial. Por lo tanto, no es posible confirmar que todos esos usuarios abandonaron la compra en esa etapa ni determinar por qué faltan algunos eventos.

Mi recomendación sería revisar primero el seguimiento de eventos y, después, analizar con mayor detalle qué sucede durante el checkout.

### ¿El cambio en el checkout mejoró la conversión?

En el experimento A/B, el grupo control obtuvo una tasa de conversión de 15.69 %, mientras que el grupo tratamiento alcanzó 16.29 %.

Aunque el resultado del tratamiento fue ligeramente superior, la prueba estadística obtuvo un valor p de 0.4161, mayor que el nivel de significancia establecido de 0.05.

Por lo tanto, no encontramos evidencia suficiente para afirmar que el nuevo diseño haya mejorado la conversión.

No recomendaría implementar el cambio únicamente con base en este resultado. Sería preferible continuar evaluándolo antes de tomar una decisión.

Este experimento corresponde específicamente a la modificación del checkout y sus resultados no deben interpretarse como una medición general del crecimiento de RappiPlus.

###  Recomendaciones finales

A partir de los resultados, considero que las principales prioridades deberían ser:

1. **Revisar la rentabilidad de los productos con pedidos que generan pérdidas**, comenzando por sus costos, precios y frecuencia de operaciones no rentables.

2. **Investigar el proceso de checkout y el registro de eventos**, para comprender mejor dónde existen posibles dificultades y obtener mediciones de conversión más confiables.

3. **Dar seguimiento a la actividad de los usuarios durante las primeras semanas**, con el fin de identificar oportunidades de retención y reactivación.

También mantendría el uso del dashboard para revisar periódicamente los ingresos, costos y márgenes por país y categoría.

###  Limitaciones y cierre

Los resultados deben interpretarse considerando algunas limitaciones.

La ganancia calculada es una estimación y no incluye todos los gastos operativos. Además, existen 30 pedidos sin costos de producto identificados.

El embudo se construyó a partir de conteos de usuarios únicos por evento, por lo que no permite reconstruir completamente el recorrido de cada usuario. La retención mide actividad semanal, no permanencia continua.

Por último, el experimento A/B evalúa un cambio específico en el checkout y no permite sacar conclusiones sobre el desempeño general del negocio.

En conjunto, el análisis muestra que RappiPlus genera una ganancia estimada positiva, pero todavía existen oportunidades para revisar la rentabilidad de ciertos productos y comprender mejor el comportamiento de los usuarios.

**La principal conclusión que me llevo del proyecto es que evaluar un negocio no consiste solamente en comprobar si sus ingresos son altos o si una modificación parece mejorar los resultados. También es necesario revisar los costos, identificar dónde se presentan las diferencias y reconocer cuándo los datos no son suficientes para llegar a una conclusión.**


---

## 🚀 Entrega Final

Comparte el acceso a tu Dashboard para revisión.   
Puedes entregar el Dashboard utilizando **Power BI o Tableau**.

Incluye **uno de los siguientes**:

- 🔗 Link público del dashboard publicado en **Power BI Service o Tableau Public / Tableau Cloud**
- 🔗 Link de **Google Drive o OneDrive** con el archivo del proyecto (`.pbix`) y los 3 csvs limpios.


### 📎 Enlace del Dashboard

# (Pega aquí tu link)
# link de power bi o tableau
# link de one drive / google drive

https://drive.google.com/drive/folders/163ceNeMOscxKID--_tldWvmckGKK2KLs?usp=sharing